# DP1 DIASource Characterization

*Notebook written by Zach Gillis (zgillis@stanford.edu)*

*Date last updated: March 9, 2026*

This is the first of three notebooks to prepare the *First Cut* filter to select lensed-AGN candidates from the Rubin Observatory alert stream using difference image analysis sources (DIASources).

This notebook examines the Rubin Observatory's Data Preview 1 (DP1) dataset and applies a **simple cuts** baseline selection to search for candidate Lensed AGN (LAGN). The simple cuts filter is a threshold-based control to compare against the XGBoost classifier in `3_lagn_classifier.ipynb`. We expect that there are no lensed-AGN in DP1, so DP1 DIASources that pass the simple cuts filter can be considered "lensed-AGN imposters." This notebook series is meant to be run on the cloud-based Rubin Science Platform JupyterLab server. (This was last tested on RSP release 29.2.0).

## Overview

Rubin's difference imaging pipeline subtracts a template image from each visit image, producing a "difference image" where only flux changes appear. Sources detected in these difference images are called DIASources. The full DP1 dataset contains 1.4 million DIASources within our three target survey fields. Our goal is to progressively filter this sample down to sources consistent with spatially extended flux differences that could be lensed AGN.

Lensed AGN are predicted to appear and be efficiently detected as spatially extended sources in difference images from optical imaging surveys (Kochanek et al. 2006). This motivates our use of extendedness metrics as the primary discriminating features in the simple cuts filter.

## Summary

1. **Load data** — read DP1 DIASources from FITS files (or query the TAP service) for all three survey fields, compute engineered features
2. **Apply simple cuts filter** — threshold-based extendedness selection
3. **Visualize** — scatter plots, corner plots, and an image gallery of surviving candidates

## Survey Fields

| Field | RA | Dec | Notes |
|---|---|---|---|
| ECDFS | 53.16° | −28.10° | Extended Chandra Deep Field South |
| Galactic | 95.0° | −25.0° | Low galactic latitude field |
| Ecliptic | 37.98° | 7.015° | Low ecliptic latitude field |

---
## Section 1 — Setup & Data Loading

### Import Libraries

In [ ]:
from lsst.rsp import get_tap_service
from lsst.rsp.service import get_siav2_service
from lsst.rsp.utils import get_pyvo_auth

import lsst.afw.display as afwDisplay
from lsst.afw.image import ExposureF
from lsst.afw.math import Warper, WarperConfig
from lsst.afw.fits import MemFileManager
import lsst.geom as geom

from pyvo.dal.adhoc import DatalinkResults, SodaQuery

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from astropy import units as u
from astropy.table import Table, vstack
import corner
from glob import glob
from astropy.coordinates import SkyCoord

import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import random
import yaml

service = get_tap_service("tap")
assert service is not None

sia_service = get_siav2_service("dp1")
assert sia_service is not None

afwDisplay.setDefaultBackend('matplotlib')

import data_processing as dp

### Load DP1 DIASources

`dp.load_dp1` reads data from all three survey fields and merges them into a single Astropy `Table`.

**Options:**
- `fetch_from_server=False` — load from local `.fits` files in `dp1_fields/` (fast, recommended)
- `fetch_from_server=True` — query the TAP service directly (slow, use only to refresh data)
- `load_all_fields=True` — load all three survey fields
- `load_all_fields=False` + `field=<field>` — load a single field; `field` options: `'ecdfs'`, `'galactic'`, `'ecliptic'`

### Engineered Features

After loading, `dp.add_engineered_features` appends derived columns used throughout this notebook:

**Notation:** $F_\text{ap}$ = `apFlux`, $F_\text{psf}$ = `psfFlux`, $F_\text{sci}$ = `scienceFlux`, $F_\text{template}$ = `template_flux` (derived); $I_{xx}, I_{yy}, I_{xy}$ = `ixx, iyy, ixy`; $I_{xx}^\text{PSF}, I_{yy}^\text{PSF}, I_{xy}^\text{PSF}$ = `ixxPSF, iyyPSF, ixyPSF`

| Column | Formula | Meaning |
|---|---|---|
| `flux_ext` | $F_\text{ap} / F_\text{psf}$ | Aperture-to-PSF flux ratio |
| `ellip_ext` | $\frac{\sqrt{(I_{xx}-I_{yy})^2+4I_{xy}^2}}{I_{xx}+I_{yy}} - \frac{\sqrt{(I_{xx}^\text{PSF}-I_{yy}^\text{PSF})^2+4(I_{xy}^\text{PSF})^2}}{I_{xx}^\text{PSF}+I_{yy}^\text{PSF}}$ | Difference of source and PSF ellipticities |
| `i_ext` | $(I_{xx}+I_{yy}) / (I_{xx}^\text{PSF}+I_{yy}^\text{PSF})$ | Source-to-PSF second moment size ratio |
| `template_flux` | $F_\text{sci} - F_\text{psf}$ | Estimated template PSF flux |
| `temp_sci_flux_ratio` | $F_\text{template} / F_\text{sci}$ | Allows removal of moving objects |
| `psf_fwhm` | $(I_{xx}^\text{PSF} I_{yy}^\text{PSF} - I_{xy}^{\text{PSF}^2})^{1/4} \times 2.355 \times 5$ | PSF FWHM in pixels |

In [ ]:
dp1_diasources = dp.load_dp1(fetch_from_server=False, load_all_fields=True, service=service)
dp1_diasources = dp.add_engineered_features(dp1_diasources)

### Dataset Summary Plots

Three plots giving a high-level overview of the raw DP1 dataset before any filtering:

1. **Visits per band**
2. **DIASources per band**
3. **DIASources per visit distribution**

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Rectangle

SHOW_INSET = False

fields = [f for f in ['ecdfs', 'galactic', 'ecliptic'] if f in set(dp1_diasources['field'])]

for field in fields:
    field_data = dp1_diasources[dp1_diasources['field'] == field]

    unique_visits_per_band = {}
    sources_per_band = {}
    sources_per_visit_by_band = {}

    for band_name in np.unique(field_data['band']):
        band_mask = field_data['band'] == band_name
        unique_visits_per_band[band_name] = len(np.unique(field_data['visit'][band_mask]))
        sources_per_band[band_name] = int(np.sum(band_mask))
        band_subset = field_data[band_mask]
        unique_visits = np.unique(band_subset['visit'])
        sources_per_visit_by_band[band_name] = np.array([
            int(np.sum(band_subset['visit'] == v)) for v in unique_visits
        ])

    total_visits = sum(unique_visits_per_band.values())
    total_sources = sum(sources_per_band.values())
    bands = [b for b in dp.BAND_ORDER if b in unique_visits_per_band]
    colors = [dp.BAND_COLORS[b] for b in bands]

    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
    fig.suptitle(f'DP1 Information — {dp.FIELD_TITLES.get(field, field)}', fontsize=16, fontweight='bold')

    # Visits per band
    ax1 = fig.add_subplot(gs[0, 0])
    visits = [unique_visits_per_band[b] for b in bands]
    bars1 = ax1.bar(bands, visits, color=colors, alpha=0.7, edgecolor='black')
    ax1.set_xlabel('Band', fontsize=12)
    ax1.set_ylabel('Number of Visits', fontsize=12)
    ax1.set_title('Number of Visits per Band', fontsize=13, fontweight='bold')
    ax1.set_ylim(0, max(visits) * 1.25)
    ax1.grid(axis='y', alpha=0.3)
    for bar in bars1:
        h = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width() / 2., h,
                 f'{int(h)}', ha='center', va='bottom', fontsize=10)
    ax1.text(0.98, 0.98, f'Total: {total_visits}', transform=ax1.transAxes,
             fontsize=11, va='top', ha='right',
             bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.7))

    # Sources per band
    ax2 = fig.add_subplot(gs[0, 1])
    sources = [sources_per_band[b] for b in bands]
    bars2 = ax2.bar(bands, sources, color=colors, alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Band', fontsize=12)
    ax2.set_ylabel('Number of DIA Sources', fontsize=12)
    ax2.set_title('DIA Sources per Band', fontsize=13, fontweight='bold')
    ax2.set_ylim(0, max(sources) * 1.25)
    ax2.grid(axis='y', alpha=0.3)
    for bar in bars2:
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width() / 2., h,
                 f'{int(h)}', ha='center', va='bottom', fontsize=10)
    ax2.text(0.98, 0.98, f'Total: {total_sources}', transform=ax2.transAxes,
             fontsize=11, va='top', ha='right',
             bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.7))

    # Sources per visit distribution
    ax3 = fig.add_subplot(gs[1, :])
    all_data = [sources_per_visit_by_band[b] for b in bands]
    stats_text = []
    for band in bands:
        d = sources_per_visit_by_band[band]
        stats_text.append(f'{band}: μ={np.mean(d):.1f}, σ={np.std(d):.1f}')

    bin_width = 100
    all_combined = np.concatenate(all_data)
    bins = np.arange(0, np.max(all_combined) + bin_width, bin_width)
    ax3.hist(all_data, bins=bins, label=bands, color=colors, alpha=0.7,
             edgecolor='black', stacked=True)
    ax3.set_xlabel('DIA Sources per Visit', fontsize=12)
    ax3.set_ylabel('Frequency', fontsize=12)
    ax3.set_title('Distribution of DIA Sources per Visit', fontsize=13, fontweight='bold')
    ax3.set_xlim(0, 5000)
    ax3.set_ylim(0, 250)
    ax3.legend(loc='upper right', fontsize=10)
    ax3.grid(axis='y', alpha=0.3)
    ax3.text(0.98, 0.4, '\n'.join(stats_text), transform=ax3.transAxes,
             fontsize=10, va='center', ha='right',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    if SHOW_INSET:
        axins = inset_axes(ax3, width="35%", height="45%", loc='upper center',
                           bbox_to_anchor=(0, 0, 1, 1), bbox_transform=ax3.transAxes)
        axins.hist(all_data, bins=bins, label=bands, color=colors, alpha=0.7,
                   edgecolor='black', stacked=True)
        axins.set_xlim(500, 5000)
        axins.set_ylim(0, 50)
        axins.set_xlabel('Sources/Visit', fontsize=9)
        axins.set_ylabel('Frequency', fontsize=9)
        axins.tick_params(labelsize=8)
        axins.grid(axis='y', alpha=0.3)
        rect = Rectangle((500, 0), 4500, 50, linewidth=1.5,
                         edgecolor='red', facecolor='none', linestyle='--')
        ax3.add_patch(rect)

    plt.show()

---
## Section 2 — Simple Cuts Filtering

This section applies the **simple cuts** filter — a sequence of threshold-based cuts to remove DIASources that are unlikely to be real astrophysical sources (quality filters) and to narrow the sample to extended DIASources (science filters).

All thresholds are loaded from `simple_cuts_thresholds.yaml`:

| Stage | Purpose |
|---|---|
| SNR cut | Remove faint, noisy detections. Currently set to SNR > 5, matching the Rubin difference imaging pipeline's own detection threshold (i.e., it doesn't do anything right now). |
| Extendedness | Keep only spatially extended detections. |
| Moving object | Remove moving objects (asteroids, etc.) that appear in the science image but not in the template. |
| Flux cap | Remove very bright sources. The per-band flux cap is set at the 99th percentile of Padma's LAGN samples. |

After all simple cuts, we expect to reduce from \~1.4M DIASources to \~100,000 LAGN imposters.

In [ ]:
with open('simple_cuts_thresholds.yaml') as f:
    simple_cuts_thresholds = yaml.safe_load(f)

sel_ext = (
    (dp1_diasources['flux_ext'] > simple_cuts_thresholds['flux_ext']) &
    (dp1_diasources['i_ext'] > simple_cuts_thresholds['i_ext']) &
    (dp1_diasources['ellip_ext'] > simple_cuts_thresholds['ellip_ext'])
)

sel_moving = dp1_diasources['temp_sci_flux_ratio'] > simple_cuts_thresholds['temp_sci_flux_ratio']

flux_cap_array = np.array([simple_cuts_thresholds['flux_caps'][band] for band in dp1_diasources['band']])
sel_flux_cap = dp1_diasources['template_flux'] < flux_cap_array

dp1_diasources_ext = dp1_diasources[sel_ext & sel_moving & sel_flux_cap]

print(len(dp1_diasources_ext))

### Visualization: Corner Plot of Extendedness Metrics

The corner plot shows all pairwise correlations between the three engineered extendedness features and the Rubin `extendedness` score.

**Columns:**

| Column | Description |
|---|---|
| `flux_ext` | `apFlux / psfFlux` — aperture-to-PSF flux ratio (log-scaled axis) |
| `ellip_ext` | Ellipticity difference between source and PSF |
| `i_ext` | Second-moment size ratio: `(Ixx+Iyy) / (IxxPSF+IyyPSF)` |
| `extendedness` | Rubin pipeline-computed extendedness score (0 = point source, 1 = extended) |

In [ ]:
columns_corner = ['flux_ext', 'ellip_ext', 'i_ext', 'extendedness']
axes_scale = ['log', 'linear', 'linear', 'linear']
ranges = [(0.1, 10.0), (0, 1), (0, 4), (0, 1)]
data_array = [dp1_diasources[col] for col in columns_corner]
labels = [
    'Flux Ext.',
    'Ellip. Diff.',
    'Moment Ext.',
    'Extendedness'
]

fig = corner.corner(np.array(data_array).T, 
                    labels=labels,
                    axes_scale=axes_scale,
                    range=ranges,
                    fill_contours=True, 
                    smooth=0.2, 
                    show_titles=False, 
                    color='grey',
                    plot_datapoints=False,
                    plot_contours=True,
                    plot_density=True,
                    bins=30)

# # Show individual points
# axes = np.array(fig.axes).reshape((len(columns_corner), len(columns_corner)))
# for i in range(len(columns_corner)):
#     for j in range(i):
#         ax = axes[i, j]
#         ax.scatter(data_array[j], data_array[i], s=2, alpha=0.03, color='black', rasterized=True)

fig.suptitle(f'Extendedness Parameter Correlations, {dp.get_title(dp1_diasources)}', fontsize=18, fontweight='bold', y=0.99)

# Text box moved up and left with padding
equation_text = (
    r'$\mathrm{Flux\ Ext.} = \log(\mathrm{Aperture\ flux} / \mathrm{PSF\ flux})$' + '\n\n' +
    r'$\mathrm{Moment\ Ext.} = \frac{I_{xx} + I_{yy}}{I_{xx}^{\mathrm{PSF}} + I_{yy}^{\mathrm{PSF}}}$' + '\n\n' +
    r'$\mathrm{Ellip.\ Diff.} = \frac{\sqrt{(I_{xx}-I_{yy})^2+4I_{xy}^2}}{I_{xx}+I_{yy}} - '
    r'\frac{\sqrt{(I_{xx}^{\mathrm{PSF}}-I_{yy}^{\mathrm{PSF}})^2+4I_{xy}^{\mathrm{PSF}\ 2}}}'
    r'{I_{xx}^{\mathrm{PSF}}+I_{yy}^{\mathrm{PSF}}}$'
)

fig.text(0.55, 0.82, equation_text, fontsize=13,
         bbox=dict(boxstyle='round,pad=0.8', facecolor='white', edgecolor='black', alpha=0.9),
         verticalalignment='top')

plt.show()

### Final Filtered Sample

`dp1_diasources_ext` is the DIASource table after all quality and science filters. 

This table is passed directly to the image gallery in Section 3, and can be considered a sample of "lensed-AGN imposters."

In [ ]:
dp1_diasources_ext

---
## Section 3 — Image Gallery

Visual inspection of a random sample of sources from `dp1_diasources_ext`. The surviving DP1 DIASources are extended sources that are non-LAGN. 

### LSST Difference Imaging Triplets + Legacy Survey Cutouts

`dp.create_image_gallery` fetches image cutouts from the Rubin archive and displays each source as a triplet:

| Panel | Image | What it shows |
|---|---|---|
| Left | Science image | The raw visit image at this location |
| Center | Template image | The deep co-add used as the subtraction reference |
| Right | Difference image | Science − Template; only flux changes appear here |

**Annotations** (shown on each panel):
- Visit ID, SNR, and science flux (left)
- RA/Dec coordinates and template flux (center)
- DIASourceID, PSF/aperture fluxes, and all extendedness metrics (right)
- Cyan circle on the difference panel = PSF FWHM, for visual size reference

Additionally, **Legacy Survey DR9 cutouts** (optical color imaging) are shown in a separate figure for the same positions. (Note: Not all fields are present in the Legacy Survey.)

**Parameters to adjust:**
- `rows`, `cols` — change the gallery layout (total images = rows × cols)


> **Note:** This cell makes live HTTP requests to the Rubin archive and Legacy Survey servers. It may take several minutes depending on the number of images and server load.

In [ ]:
fig_lsst, fig_legacy = dp.create_image_gallery(dp1_diasources_ext, rows=2, cols=2)
plt.show()